# LexAR - Fases 1 y 2

Notebook de trabajo para implementar las primeras dos fases del proyecto:

1. **Consolidacion de datos:** cargar el corpus procesado, unir metadatos y segmentar textos normativos en fragmentos comparables.
2. **Embeddings y busqueda:** generar representaciones vectoriales iniciales, recuperar vecinos cercanos y armar candidatos para el analisis posterior de redundancias/contradicciones.

La salida de este notebook no afirma contradicciones juridicas. Produce candidatos semanticamente cercanos para revisar en la fase siguiente con un LLM o evaluacion humana.

## 0. Dependencias

Si el entorno no tiene `pyarrow`, ejecutar la celda siguiente. `pyarrow` es necesario para leer el `.parquet` del corpus.

In [ ]:
# Ejecutar solo si faltan dependencias.
# %pip install pandas pyarrow numpy

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import re
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

if importlib.util.find_spec("pyarrow") is None:
    raise RuntimeError("Falta pyarrow. Ejecutar la celda de dependencias: %pip install pandas pyarrow numpy")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 220)

## 1. Configuracion

Los parametros estan pensados para poder correr una primera prueba en una notebook sin procesar todo el corpus. Para escalar, subir `MAX_TEXT_VERSIONS` y `MAX_FRAGMENTS_FOR_EMBEDDINGS` gradualmente.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    # Soporta lanzar Jupyter tanto desde la raiz del repo como desde notebooks/.
    ROOT = ROOT.parent

DATASET_DIR = ROOT / "data" / "lexar_datos_infoleg_saij"
TEXT_VERSIONS_PATH = DATASET_DIR / "corpus_unificado" / "text_versions.parquet"
DOCUMENTS_PATH = DATASET_DIR / "infoleg" / "procesado" / "documents.csv"
RELATIONS_PATH = DATASET_DIR / "infoleg" / "procesado" / "relations.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Muestra de textos normativos a segmentar. None procesa todos los textos, pero puede tardar.
MAX_TEXT_VERSIONS = 1_000

# Limite de fragmentos que entran a embeddings/busqueda en esta corrida.
MAX_FRAGMENTS_FOR_EMBEDDINGS = 5_000

# Segmentacion.
MIN_FRAGMENT_CHARS = 80
MAX_FRAGMENT_CHARS = 2_500
CHUNK_OVERLAP_CHARS = 250

# Embeddings locales por hashing. Es un baseline reproducible/offline.
N_HASH_FEATURES = 512
TOP_K_NEIGHBORS = 8
SIMILARITY_THRESHOLD_FOR_CLUSTERS = 0.35
RANDOM_SEED = 42

assert TEXT_VERSIONS_PATH.exists(), TEXT_VERSIONS_PATH
assert DOCUMENTS_PATH.exists(), DOCUMENTS_PATH

TEXT_VERSIONS_PATH, DOCUMENTS_PATH, OUTPUT_DIR

## Fase 1 - Consolidacion de datos

Se carga la tabla unificada de textos (`text_versions.parquet`) y se une con los metadatos principales de Infoleg. La tabla resultante conserva trazabilidad hacia `document_id`, fuente, version y datos bibliograficos basicos.

In [ ]:
text_versions = pd.read_parquet(TEXT_VERSIONS_PATH)
documents = pd.read_csv(DOCUMENTS_PATH, dtype=str, keep_default_na=False)

print(f"Text versions: {len(text_versions):,}")
print(f"Documentos metadata: {len(documents):,}")

display(text_versions.head(3))
display(documents.head(3))

In [ ]:
metadata_cols = [
    "document_id",
    "tipo_norma",
    "clase_norma",
    "fecha_sancion",
    "numero_boletin",
    "fecha_boletin",
    "titulo_resumido",
    "titulo_sumario",
    "texto_resumido",
]

corpus = text_versions.merge(documents[metadata_cols], on="document_id", how="left")

if MAX_TEXT_VERSIONS is not None and len(corpus) > MAX_TEXT_VERSIONS:
    corpus_sample = corpus.sample(MAX_TEXT_VERSIONS, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    corpus_sample = corpus.reset_index(drop=True)

coverage = {
    "text_versions_total": int(len(text_versions)),
    "documents_total": int(len(documents)),
    "corpus_rows_after_join": int(len(corpus)),
    "corpus_sample_rows": int(len(corpus_sample)),
    "unique_documents_in_sample": int(corpus_sample["document_id"].nunique()),
    "sources_in_sample": corpus_sample["text_source"].value_counts().to_dict(),
}

print(json.dumps(coverage, indent=2, ensure_ascii=False))

### 1.1 Segmentacion en articulos y chunks

La unidad ideal es el articulo. Cuando el patron de articulos no aparece o un articulo queda demasiado largo, el notebook usa chunks con solapamiento como fallback. Esto evita perder textos largos como codigos o compilaciones.

In [ ]:
ARTICLE_PATTERN = re.compile(
    r"(?m)^\s*(?:\*+\s*)?ARTICULO\s+([0-9]+|[IVXLCDM]+|[A-Z]+)(?:\s*[.º°-])?",
)

TOKEN_PATTERN = re.compile(r"[a-zA-ZáéíóúÁÉÍÓÚñÑüÜ0-9]{3,}")


def normalize_for_article_detection(text: str) -> str:
    """Normaliza solo para detectar encabezados; conserva largo y posiciones."""
    return (
        text.upper()
        .replace("Á", "A")
        .replace("É", "E")
        .replace("Í", "I")
        .replace("Ó", "O")
        .replace("Ú", "U")
        .replace("Ü", "U")
    )


def clean_fragment_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


def chunk_text(text: str, max_chars: int = MAX_FRAGMENT_CHARS, overlap: int = CHUNK_OVERLAP_CHARS):
    text = str(text)
    if len(text) <= max_chars:
        yield 0, len(text), clean_fragment_text(text)
        return

    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = max(text.rfind(". ", start, end), text.rfind("\n", start, end))
            if boundary > start + max_chars * 0.5:
                end = boundary + 1
        chunk = clean_fragment_text(text[start:end])
        if len(chunk) >= MIN_FRAGMENT_CHARS:
            yield start, end, chunk
        if end >= len(text):
            break
        start = max(0, end - overlap)


def segment_text_version(row: pd.Series) -> list[dict]:
    text = str(row.get("full_text", ""))
    normalized = normalize_for_article_detection(text)
    matches = list(ARTICLE_PATTERN.finditer(normalized))
    fragments = []

    if len(matches) >= 2:
        spans = []
        for index, match in enumerate(matches):
            start = match.start()
            end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
            label = f"ARTICULO {match.group(1)}"
            spans.append((start, end, label))

        for article_index, (start, end, label) in enumerate(spans, start=1):
            article_text = clean_fragment_text(text[start:end])
            if len(article_text) < MIN_FRAGMENT_CHARS:
                continue
            if len(article_text) <= MAX_FRAGMENT_CHARS:
                fragments.append({
                    "fragment_type": "article",
                    "ordinal": article_index,
                    "label": label,
                    "char_start": start,
                    "char_end": end,
                    "text": article_text,
                })
            else:
                for chunk_index, (chunk_start, chunk_end, chunk) in enumerate(chunk_text(article_text), start=1):
                    fragments.append({
                        "fragment_type": "article_chunk",
                        "ordinal": article_index,
                        "label": f"{label} / CHUNK {chunk_index}",
                        "char_start": start + chunk_start,
                        "char_end": start + chunk_end,
                        "text": chunk,
                    })
    else:
        for chunk_index, (start, end, chunk) in enumerate(chunk_text(text), start=1):
            fragments.append({
                "fragment_type": "chunk",
                "ordinal": chunk_index,
                "label": f"CHUNK {chunk_index}",
                "char_start": start,
                "char_end": end,
                "text": chunk,
            })

    for fragment in fragments:
        fragment.update({
            "text_version_id": row.get("text_version_id"),
            "document_id": row.get("document_id"),
            "text_source": row.get("text_source"),
            "version_type": row.get("version_type"),
            "fecha_sancion": row.get("fecha_sancion"),
            "tipo_norma": row.get("tipo_norma"),
            "titulo_resumido": row.get("titulo_resumido"),
            "source_url": row.get("source_url"),
        })
        content_hash = hashlib.sha256(fragment["text"].encode("utf-8", errors="ignore")).hexdigest()
        fragment["content_hash"] = content_hash
        fragment["text_len"] = len(fragment["text"])

    return fragments

In [ ]:
all_fragments = []

for _, row in corpus_sample.iterrows():
    all_fragments.extend(segment_text_version(row))

fragments = pd.DataFrame(all_fragments)
if not fragments.empty:
    fragments = fragments.reset_index(drop=True)
    fragments.insert(0, "fragment_id", [f"frag:{i:08d}" for i in range(len(fragments))])

print(f"Fragmentos generados: {len(fragments):,}")
display(fragments.head(10))

In [ ]:
fragment_profile = {
    "fragments_total": int(len(fragments)),
    "documents_total": int(fragments["document_id"].nunique()) if not fragments.empty else 0,
    "by_fragment_type": fragments["fragment_type"].value_counts().to_dict() if not fragments.empty else {},
    "text_len_stats": fragments["text_len"].describe().round(2).to_dict() if not fragments.empty else {},
}

print(json.dumps(fragment_profile, indent=2, ensure_ascii=False))

fragments_path = OUTPUT_DIR / "legal_fragments_sample.csv"
fragments.to_csv(fragments_path, index=False, encoding="utf-8")
fragments_path

## Fase 2 - Embeddings y busqueda

Para que el notebook sea reproducible sin red, se implementa un baseline de embeddings locales por hashing de tokens. Esto no reemplaza un embedding semantico de alta calidad, pero permite probar toda la tuberia: vectorizar, buscar vecinos y generar candidatos.

Luego se puede reemplazar esta celda por embeddings de API u otro modelo multilingue.

In [ ]:
def tokenize(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(str(text))]


def stable_hash_index(token: str, n_features: int) -> int:
    digest = hashlib.blake2b(token.encode("utf-8", errors="ignore"), digest_size=8).digest()
    return int.from_bytes(digest, "little") % n_features


def hashing_embeddings(texts: list[str], n_features: int = N_HASH_FEATURES) -> np.ndarray:
    matrix = np.zeros((len(texts), n_features), dtype=np.float32)
    document_frequency = np.zeros(n_features, dtype=np.float32)

    tokenized = []
    for text in texts:
        tokens = tokenize(text)
        tokenized.append(tokens)
        for index in {stable_hash_index(token, n_features) for token in tokens}:
            document_frequency[index] += 1

    idf = np.log((1 + len(texts)) / (1 + document_frequency)) + 1

    for row_index, tokens in enumerate(tokenized):
        if not tokens:
            continue
        for token in tokens:
            matrix[row_index, stable_hash_index(token, n_features)] += 1
        matrix[row_index] *= idf

    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return matrix / norms


embedding_fragments = fragments.copy()
if MAX_FRAGMENTS_FOR_EMBEDDINGS is not None and len(embedding_fragments) > MAX_FRAGMENTS_FOR_EMBEDDINGS:
    embedding_fragments = embedding_fragments.sample(MAX_FRAGMENTS_FOR_EMBEDDINGS, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Fragmentos para embeddings: {len(embedding_fragments):,}")

embeddings = hashing_embeddings(embedding_fragments["text"].tolist())
embeddings.shape

### 2.1 Opcion: embeddings por API

Esta celda queda como reemplazo opcional del baseline local. Requiere configurar `OPENAI_API_KEY` y tener instalado el paquete `openai`. No se ejecuta por defecto para evitar costos accidentales.

In [ ]:
# Opcional: descomentar para usar embeddings de API.
# %pip install openai
# from openai import OpenAI
# client = OpenAI()
#
# def openai_embeddings(texts, model="text-embedding-3-small", batch_size=100):
#     vectors = []
#     for start in range(0, len(texts), batch_size):
#         batch = texts[start:start + batch_size]
#         response = client.embeddings.create(model=model, input=batch)
#         vectors.extend([item.embedding for item in response.data])
#     matrix = np.array(vectors, dtype=np.float32)
#     norms = np.linalg.norm(matrix, axis=1, keepdims=True)
#     norms[norms == 0] = 1
#     return matrix / norms
#
# embeddings = openai_embeddings(embedding_fragments["text"].tolist())

### 2.2 Recuperacion de vecinos cercanos

Se calcula similitud coseno entre embeddings normalizados. Para evitar candidatos triviales, se excluyen pares del mismo `document_id`.

In [ ]:
def top_neighbors(embeddings: np.ndarray, metadata: pd.DataFrame, top_k: int = TOP_K_NEIGHBORS, block_size: int = 256) -> pd.DataFrame:
    rows = []
    doc_ids = metadata["document_id"].tolist()

    for start in range(0, len(metadata), block_size):
        end = min(start + block_size, len(metadata))
        scores = embeddings[start:end] @ embeddings.T

        for local_index, global_index in enumerate(range(start, end)):
            score_row = scores[local_index]
            score_row[global_index] = -np.inf

            same_document = np.array([doc_id == doc_ids[global_index] for doc_id in doc_ids])
            score_row[same_document] = -np.inf

            candidate_indices = np.argpartition(-score_row, range(min(top_k, len(score_row))))[:top_k]
            candidate_indices = candidate_indices[np.argsort(-score_row[candidate_indices])]

            for neighbor_index in candidate_indices:
                similarity = float(score_row[neighbor_index])
                if not math.isfinite(similarity):
                    continue
                a = metadata.iloc[global_index]
                b = metadata.iloc[neighbor_index]
                pair_key = tuple(sorted([a["fragment_id"], b["fragment_id"]]))
                rows.append({
                    "pair_key": "|".join(pair_key),
                    "fragment_a_id": a["fragment_id"],
                    "fragment_b_id": b["fragment_id"],
                    "document_a_id": a["document_id"],
                    "document_b_id": b["document_id"],
                    "label_a": a["label"],
                    "label_b": b["label"],
                    "title_a": a.get("titulo_resumido", ""),
                    "title_b": b.get("titulo_resumido", ""),
                    "date_a": a.get("fecha_sancion", ""),
                    "date_b": b.get("fecha_sancion", ""),
                    "similarity_score": similarity,
                    "text_a": a["text"],
                    "text_b": b["text"],
                })

    candidates = pd.DataFrame(rows)
    if candidates.empty:
        return candidates

    candidates = (
        candidates.sort_values("similarity_score", ascending=False)
        .drop_duplicates("pair_key")
        .reset_index(drop=True)
    )
    candidates.insert(0, "candidate_id", [f"cand:{i:08d}" for i in range(len(candidates))])
    candidates["retrieval_method"] = "hashing_tfidf_cosine"
    return candidates


candidates = top_neighbors(embeddings, embedding_fragments)
print(f"Candidatos recuperados: {len(candidates):,}")

display(candidates[[
    "candidate_id", "similarity_score", "document_a_id", "label_a", "title_a", "document_b_id", "label_b", "title_b"
]].head(20))

In [ ]:
candidates_path = OUTPUT_DIR / "analysis_candidates_sample.csv"
candidates.to_csv(candidates_path, index=False, encoding="utf-8")
candidates_path

### 2.3 Clusters exploratorios

Como primer acercamiento a los subespacios semanticos, armamos componentes conectadas entre fragmentos con similitud mayor a un umbral. No es clustering definitivo; sirve para inspeccionar grupos de normas potencialmente relacionadas.

In [ ]:
def connected_components_from_candidates(candidates: pd.DataFrame, threshold: float) -> pd.DataFrame:
    graph = defaultdict(set)
    strong_edges = candidates[candidates["similarity_score"] >= threshold]

    for _, row in strong_edges.iterrows():
        graph[row["fragment_a_id"]].add(row["fragment_b_id"])
        graph[row["fragment_b_id"]].add(row["fragment_a_id"])

    visited = set()
    clusters = []
    cluster_id = 0

    for node in graph:
        if node in visited:
            continue
        queue = deque([node])
        visited.add(node)
        component = []
        while queue:
            current = queue.popleft()
            component.append(current)
            for neighbor in graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        if len(component) >= 2:
            cluster_id += 1
            clusters.append({
                "cluster_id": f"cluster:{cluster_id:04d}",
                "size": len(component),
                "fragment_ids": component,
            })

    return pd.DataFrame(clusters).sort_values("size", ascending=False).reset_index(drop=True) if clusters else pd.DataFrame(columns=["cluster_id", "size", "fragment_ids"])


clusters = connected_components_from_candidates(candidates, SIMILARITY_THRESHOLD_FOR_CLUSTERS)
print(f"Clusters exploratorios: {len(clusters):,}")
display(clusters.head(20))

In [ ]:
cluster_rows = []
fragment_lookup = embedding_fragments.set_index("fragment_id")

for _, cluster in clusters.head(10).iterrows():
    for fragment_id in cluster["fragment_ids"][:10]:
        fragment = fragment_lookup.loc[fragment_id]
        cluster_rows.append({
            "cluster_id": cluster["cluster_id"],
            "cluster_size": cluster["size"],
            "fragment_id": fragment_id,
            "document_id": fragment["document_id"],
            "label": fragment["label"],
            "titulo_resumido": fragment.get("titulo_resumido", ""),
            "text_preview": fragment["text"][:500],
        })

cluster_preview = pd.DataFrame(cluster_rows)
display(cluster_preview)

## Resultado de fases 1 y 2

Este notebook deja implementado:

- carga y union del corpus procesado con metadatos;
- segmentacion inicial en articulos/chunks;
- export de `outputs/legal_fragments_sample.csv`;
- embeddings locales reproducibles;
- recuperacion de vecinos cercanos;
- export de `outputs/analysis_candidates_sample.csv`;
- clusters exploratorios para inspeccion.

La fase siguiente deberia tomar `analysis_candidates_sample.csv` y clasificar cada par con un prompt juridico/PLN que devuelva `possible_conflict`, `possible_overlap`, `different_scope`, `neutral` o `needs_review`, junto con una explicacion y una propuesta de redaccion alternativa.